# Training the Transformer imputer on Kaggle

Runs `transformer.main_train` on a Kaggle GPU and generates the submission.

1. Add the challenge CSVs (`X_train.csv`, `y_train.csv`, `X_test.csv`) as a Kaggle dataset input.
2. Enable a GPU accelerator and run all cells.

Checkpoints are written after every epoch to `transformer/checkpoints/seed<N>/` inside
`/kaggle/working`. If the session stops, save the notebook version (so `/kaggle/working`
is kept), set `RESUME = True` and run again.

In [ ]:
SEED = 42
RESUME = False
REPO = "https://github.com/PolTibau/DataChallenge.git"
ROOT = "/kaggle/working/DataChallenge"

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

root = Path(ROOT)
if (root / ".git").exists():
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
os.chdir(root)

# Data: link the CSVs from the Kaggle input dataset into data/
data_dir = root / "data"
data_dir.mkdir(exist_ok=True)
src = next(Path("/kaggle/input").rglob("X_train.csv")).parent
for name in ["X_train.csv", "y_train.csv", "X_test.csv"]:
    dst = data_dir / name
    if not dst.exists():
        try:
            os.symlink(src / name, dst)
        except OSError:
            shutil.copy2(src / name, dst)
print(sorted(p.name for p in data_dir.iterdir()))

In [ ]:
cmd = ["python", "-m", "transformer.main_train", "--seed", str(SEED)] + (["--resume"] if RESUME else [])
subprocess.run(cmd, check=True)

In [ ]:
subprocess.run(["python", "-m", "transformer.main_predict"], check=True)
print(sorted((root / "outputs" / "transformer").iterdir()))